# Analyzing errors by segment

_Doing ML for Real — the skills that matter (2026)_

**The average hides the failure. Slice the metric and the broken group jumps out.**

One number summarizes a whole test set. Summaries average away detail — including the detail that the model is broken on a specific group.

       Segmented error analysis is simple: split the test set into meaningful groups (segments) and recompute the metric inside each one. The aggregate is a weighted average of those per-segment numbers, so a high average can sit on top of one terrible slice.

---

This notebook is a practice scaffold for the **Analyzing errors by segment** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q fairlearn
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — fairlearn + pandas

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score
from fairlearn.metrics import MetricFrame, count

# y_true, y_pred : arrays from your held-out test set
# segment        : a categorical/ binned column, one label per row
#                  (e.g. customer tier, region, or binned mean-radius tertile)

# ---- 1) fairlearn: per-segment metrics + the worst slice, in one object ----
mf = MetricFrame(
    metrics={"accuracy": accuracy_score,
             "recall": recall_score,
             "n": count},                 # row count per group (always report it!)
    y_true=y_true,
    y_pred=y_pred,
    sensitive_features=segment,           # the slicing column
)
print(mf.overall)                         # the headline numbers
print(mf.by_group)                        # the per-segment table
print("worst recall slice:", mf.recall.idxmin())   # which group is worst
print("recall gap (max-min):", mf.difference(method="between_groups")["recall"])

# ---- 2) pandas groupby fallback (no fairlearn needed) ----
df = pd.DataFrame({"y": y_true, "pred": y_pred, "seg": segment})
per_seg = (df.groupby("seg")
             .apply(lambda g: pd.Series({
                 "n": len(g),
                 "accuracy": accuracy_score(g.y, g.pred),
                 "recall": recall_score(g.y, g.pred),
             }))
             .sort_values("recall"))
print(per_seg)                            # bottom row = worst slice to investigate

# ---- 3) is the worst slice's gap real? two-proportion z-test (slice vs rest) ----
from statsmodels.stats.proportion import proportions_ztest
worst = per_seg.index[0]
m = (segment == worst) & (y_true == 1)            # benign cases inside worst slice
r = (segment != worst) & (y_true == 1)            # benign cases everywhere else
succ = np.array([(y_pred[m] == 1).sum(), (y_pred[r] == 1).sum()])
nobs = np.array([m.sum(), r.sum()])
z, p = proportions_ztest(succ, nobs)              # |z|>1.96  => gap is significant
print(f"worst slice={worst}  z={z:.2f}  p={p:.4f}")

## Visualize the data & results

_On the breast-cancer scans, a model with 0.965 overall accuracy — but is its benign recall uniform across tumor sizes, or does one segment quietly fail?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score

d = load_breast_cancer()                       # 569 real tumor scans, 30 features
X, y = d.data, d.target                         # y=1 benign, y=0 malignant
mean_radius = X[:, 0]                            # feature 0 = mean radius

# split, carrying the mean-radius column so we can derive the segment on the test set
Xtr, Xte, ytr, yte, mr_tr, mr_te = train_test_split(
    X, y, mean_radius, test_size=0.4, random_state=0, stratify=y)

clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
pred = clf.predict(Xte)
print("overall accuracy:", round((pred == yte).mean(), 3))   # 0.965

# derive the SEGMENT: tertiles of mean radius (cut points fixed on TRAIN)
q1, q2 = np.quantile(mr_tr, [1/3, 2/3])         # ~ 12.01, 14.73
seg = np.where(mr_te <= q1, "small",
       np.where(mr_te <= q2, "medium", "large"))

# benign recall PER segment
for name in ["small", "medium", "large"]:
    m = seg == name
    rec = recall_score(yte[m], pred[m], pos_label=1)
    print(f"{name:7s} n={m.sum():3d}  benign recall={rec:.3f}")
# small   n= 57  benign recall=1.000
# medium  n= 93  benign recall=0.974
# large   n= 78  benign recall=0.750   <- worst slice (12 benign cases, 9 caught)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your churn model reports 0.92 accuracy and the team wants to ship. The product owner asks "is it good for our enterprise customers?" — who are 6% of users. What do you do before answering?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Refuse to answer from the aggregate. Define segments first: customer tier (enterprise vs SMB), and maybe tenure and region. — _0.92 is an average over all users. Enterprise is only 6%, so it can be terrible while the headline stays high._
- Compute accuracy (and recall on the churn class) per tier with a groupby, attaching a row count to each slice. — _Per-segment metrics reveal whether the enterprise slice tracks the average or lags it. The count tells you how much to trust each slice._
- If enterprise looks worse, run a two-proportion z-test of enterprise vs the rest before raising an alarm. — _On 6% of users the slice may be small; a gap on few rows could be noise. The z-test separates a real gap from sampling wobble._

**Answer:** Slice by customer tier and recompute the metric per segment with counts; if enterprise lags, confirm the gap is significant with a two-proportion z-test before trusting it. The 0.92 aggregate cannot answer the question.

</details>

**Problem 2.** You swap in model B. Overall AUC rises from 0.81 to 0.83, so it looks like a clear win. A reviewer warns about Simpson's paradox. How do you check, and what would change your decision?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Break the test set into the cohorts that matter (e.g. new vs returning users) and compute each model's AUC within each cohort. — _Simpson's paradox is when an aggregate improves while every subgroup regresses, usually because the cohort mix shifted between the two evaluations._
- Compare model A vs model B per cohort, not just overall. — _If B's per-cohort AUC is lower in every cohort despite a higher blended AUC, the global gain is an artifact of reweighting, not real improvement._
- Only ship B if its global win also holds (or at least does not regress) within each important cohort. — _A genuine improvement should survive at the subgroup level; a paradoxical one would quietly hurt real users._

**Answer:** Recompute each model's AUC per cohort. If B is worse on every cohort despite a higher overall AUC, that is Simpson's paradox — do not ship. Require the win to hold per subgroup.

</details>